In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the taxis dataset
df = sns.load_dataset("taxis")

# Display the first five rows
print(df.head())

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nNumerical summary:")
print(df.describe())

In [ ]:
# Check missing values before cleaning
print("Missing values before cleaning:")
print(df.isnull().sum())

# Convert pickup to datetime; invalid dates become missing values
df["pickup"] = pd.to_datetime(df["pickup"], errors="coerce")

# Identify numerical and categorical columns
numeric_cols = df.select_dtypes(include="number").columns
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns

# Fill numerical missing values with the median
for col in numeric_cols:
    median_value = df[col].median()
    if pd.notna(median_value):
        df[col] = df[col].fillna(median_value)

# Fill categorical missing values with the mode
for col in categorical_cols:
    mode_values = df[col].mode()
    if not mode_values.empty:
        df[col] = df[col].fillna(mode_values[0])

# Remove rows missing critical values required for the main analysis
df = df.dropna(subset=["pickup", "fare", "distance"])

print("\nMissing values after cleaning:")
print(df.isnull().sum())

print("\nDataset shape after cleaning:", df.shape)

In [ ]:
# Sort records by pickup timestamp
line_data = df.sort_values("pickup")

plt.figure(figsize=(12, 5))
plt.plot(line_data["pickup"], line_data["fare"], linewidth=1)

plt.xlabel("Pickup Timestamp")
plt.ylabel("Fare")
plt.title("Taxi Fare over Time")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
borough_fare = df.groupby("pickup_borough", observed=False)["fare"].sum().sort_values(ascending=False)

borough_fare.plot(kind="bar", figsize=(8, 5))

plt.xlabel("Pickup Borough")
plt.ylabel("Total Fare")
plt.title("Total Fare by Pickup Borough")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
payment_counts = df["payment"].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(
    payment_counts,
    labels=payment_counts.index.astype(str),
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Distribution of Trips by Payment Method")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df["distance"].dropna(), bins=20, edgecolor="black")

plt.xlabel("Trip Distance")
plt.ylabel("Number of Trips")
plt.title("Distribution of Taxi Trip Distance")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
df.boxplot(column="tip", by="pickup_borough", grid=False)

plt.xlabel("Pickup Borough")
plt.ylabel("Tip Amount")
plt.title("Distribution of Tip Amount by Pickup Borough")
plt.suptitle("")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(data=df, x="pickup_borough")

plt.xlabel("Pickup Borough")
plt.ylabel("Number of Trips")
plt.title("Number of Trips by Pickup Borough")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="distance",
    y="fare",
    hue="pickup_borough",
    alpha=0.6
)

plt.xlabel("Distance")
plt.ylabel("Fare")
plt.title("Relationship Between Distance and Fare")
plt.tight_layout()
plt.show()

In [ ]:
correlation_columns = ["distance", "fare", "tip", "tolls", "total"]
correlation_matrix = df[correlation_columns].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Heatmap of Taxi Variables")
plt.tight_layout()
plt.show()

In [ ]:
pair_columns = ["distance", "fare", "tip", "total", "pickup_zone"]
pair_data = df[pair_columns].dropna().copy()

# Keep the five most frequent pickup zones so the plot is readable
top_zones = pair_data["pickup_zone"].value_counts().head(5).index
pair_data = pair_data[pair_data["pickup_zone"].isin(top_zones)]

# Use a sample of up to 1,000 rows for plotting performance
if len(pair_data) > 1000:
    pair_data = pair_data.sample(n=1000, random_state=42)

sns.pairplot(
    pair_data,
    vars=["distance", "fare", "tip", "total"],
    hue="pickup_zone",
    diag_kind="hist"
)

plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.violinplot(
    data=df,
    x="payment",
    y="fare",
    cut=0
)

plt.xlabel("Payment Method")
plt.ylabel("Fare")
plt.title("Distribution of Fare by Payment Method")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()